# G10 — The single preregistered test evaluation

The plan is fixed in `experiments/rtt/PREREG_G10_TEST.md`, which was committed before this notebook was run. In short:

* **Training data.** All arms train on the 45 train+val episodes. Five of those episodes, drawn with
  `random.Random(2026)`, are held out for early stopping and checkpoint selection.
* **Evaluation.** Once, on the 409 test MCIS from 8 episodes.
* **Arms:** `PaperBest` (the Hi-EF paper's best configuration, replicated), `B1`, `LateFusion`, `RoleNet`,
  `RoleNet-noRole`. Each neural arm runs 5 seeds, and the seed ensemble averages probabilities.
* **Confirmatory contrasts**, in a fixed sequence, measured as ΔUAR under a single post-hoc logit adjustment (LA) with a
  95% bootstrap over the test episodes:
  1. RoleNet − PaperBest (primary)
  2. RoleNet − B1
  3. RoleNet − LateFusion
  4. RoleNet − RoleNet-noRole

Run it **once**. Set `UNLOCK_TEST = True` in the CONFIG cell.

In [ ]:
!pip install -q speechbrain

In [ ]:
# ======== CONFIG ========
import os


def first_existing(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"none of {paths}")


DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv",
                           "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv")
G8A_DIR = first_existing("/kaggle/input/datasets/ptrnghieu/g8a-features", "/kaggle/input/g8a-features")
OUT_DIR = "/kaggle/working"
UNLOCK_TEST = False          # set to True for the single preregistered run

N_INNER_DEV, SELECT_SEED = 5, 2026
SEEDS = [42, 123, 456, 789, 1024]
LR, WEIGHT_DECAY = 1e-4, 1e-5
FC_EPOCHS, PATIENCE, FC_BATCH = 50, 8, 32          # B1, as G3b / G8b
PAPER_EPOCHS, PAPER_BATCH = 50, 32                 # PaperBest, as the replication notebook
RN = dict(D=128, heads=4, layers=2, dropout=0.2, lr=3e-4, wd=1e-2, epochs=80, patience=12, batch=64,
          aux_w=0.3, a_w=0.3, p_drop_ctx=0.3, p_drop_face=0.15)
PCA_DIM, MAXF, MAXF_POOL = 128, 24, 32
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25
LATE_W, LA_TAU = 0.5, 1.0
VOICE_SAME_COS = 0.35
DEBUG_PER_EPISODE = None

FULL = dict(role=True, faces=True, ctx=True, aux=True, mdrop=True)
EXPERIMENTS = [
    ("B1",             'b1',   None),
    ("RoleNet",        'role', FULL),
    ("RoleNet-noRole", 'role', {**FULL, 'role': False}),
]

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import glob, pickle
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

if not UNLOCK_TEST:
    raise RuntimeError("Test split is locked. Set UNLOCK_TEST = True only for the single preregistered run.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF", "annotation.csv"))[0]
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
if DEBUG_PER_EPISODE:
    sp = sp.groupby('source_folder', group_keys=False).head(DEBUG_PER_EPISODE)
DEV = sp.reset_index(drop=True)            # all rows: train+val are fitted on, test is evaluated once
train_all = ev = DEV
N = len(DEV)
IS_TEST = (DEV.split == 'test').values
src = DEV.source_folder.values
EPS = np.array(sorted(set(src[~IS_TEST])))
TEST_EPS = sorted(set(src[IS_TEST]))
print(f"training MCIS {(~IS_TEST).sum()} from {len(EPS)} episodes | test MCIS {IS_TEST.sum()} from {len(TEST_EPS)} episodes")
if not DEBUG_PER_EPISODE:
    assert len(EPS) == 45 and len(TEST_EPS) == 8 and IS_TEST.sum() == 409


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ---- load every clip used by any MCIS (I-IV) once, keep it on the GPU
all_clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()) & set(
    f[:-3].replace('_', '/', 1) for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')))
CIDX = {c: i for i, c in enumerate(all_clips)}
missing = [c for c in set(train_all[['clip1', 'clip2', 'clip3']].values.ravel()) | set(ev[['clip1', 'clip2', 'clip3']].values.ravel())
           if c not in CIDX]
assert not missing, f"{len(missing)} clips without features, e.g. {missing[:3]}"

bufs = {k: [] for k in ('face', 'fmask', 'ori', 'text', 'audio', 'afound')}
for c in tqdm(all_clips, desc='loading features'):
    d = torch.load(os.path.join(FEATURES_DIR, c.replace('/', '_') + '.pt'), map_location='cpu', weights_only=False)
    face = d['face_features'].float()
    fm = d.get('face_valid_mask')
    bufs['face'].append(face)
    bufs['fmask'].append(torch.ones(face.shape[0], dtype=torch.bool) if fm is None else torch.as_tensor(fm).bool().reshape(-1))
    bufs['ori'].append(d['ori_features'].float())
    bufs['text'].append(d['text_feature'].float().reshape(-1))
    bufs['audio'].append(d.get('audio_feature', torch.zeros(527)).float().reshape(-1))
    bufs['afound'].append(torch.tensor(bool(d.get('audio_found', True))))
FEAT = {k: torch.stack(v).to(DEVICE) for k, v in bufs.items()}
del bufs
print({k: tuple(v.shape) for k, v in FEAT.items()})


def gather(idx):
    """idx: LongTensor of clip indices (any shape) -> dict of feature tensors with that leading shape."""
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, d=512, n_frames=16, layers=2, heads=8, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_frames, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

    def forward(self, x, mask):  # mask: True = valid frame
        mask = mask.clone()
        mask[~mask.any(1), 0] = True
        h = self.enc(x + self.pos[:, :x.size(1)], src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)


class ClipEncoder(nn.Module):
    """Face/original temporal encoders + text/audio tokens -> 1-layer fusion Transformer -> one 512-d vector."""

    def __init__(self, d=512):
        super().__init__()
        self.face, self.ori = TemporalEncoder(d), TemporalEncoder(d)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.modality = nn.Parameter(torch.randn(1, 4, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, b):
        ori_mask = torch.ones(b['ori'].shape[:2], dtype=torch.bool, device=b['ori'].device)
        tokens = torch.stack([self.face(b['face'], b['fmask']), self.ori(b['ori'], ori_mask),
                              self.text(b['text']), self.audio(F.normalize(b['audio'], dim=-1))], 1)
        valid = torch.ones(tokens.shape[:2], dtype=torch.bool, device=tokens.device)
        valid[:, 3] = b['afound']
        h = self.fusion(tokens + self.modality, src_key_padding_mask=~valid)
        m = valid.unsqueeze(-1).float()
        return self.norm((h * m).sum(1) / m.sum(1))


class ClipRecognizer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.drop = nn.Dropout(0.3)
        self.emo, self.pol = nn.Linear(d, 7), nn.Linear(d, 3)

    def forward(self, b):
        h = self.drop(self.enc(b))
        return self.emo(h), self.pol(h)


N_REC = 12   # 7 emotion probs + 3 polarity probs + max prob + entropy


class Forecaster(nn.Module):
    def __init__(self, use_raw=True, use_traj=False, d=512, positions=None):
        super().__init__()
        self.use_raw, self.use_traj = use_raw, use_traj
        self.positions = positions   # clip positions (0=I, 1=II, 2=III); None = the last n clips
        self.enc = ClipEncoder(d) if use_raw else None
        self.traj = nn.Sequential(nn.LayerNorm(N_REC), nn.Linear(N_REC, d), nn.GELU(), nn.Linear(d, d)) if use_traj else None
        self.clip_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.inter = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d // 2, 7))

    def forward(self, clip_idx, rec):  # clip_idx [B,n], rec [B,n,N_REC], n <= 3 clips in temporal order
        B, n = clip_idx.shape
        tok = 0
        if self.use_raw:
            feats = gather(clip_idx)
            flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
            tok = self.enc(flat).reshape(B, n, -1)
        if self.use_traj:
            tok = tok + self.traj(rec)
        pos = self.clip_pos[:, list(self.positions)] if self.positions is not None else self.clip_pos[:, 3 - n:]
        h = self.inter(tok + pos)
        return self.head(h.mean(1))

## From G8a features to role-tagged slots

In [ ]:
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

G8 = {}
for f in sorted(glob.glob(os.path.join(G8A_DIR, '**', 'shard_*.pkl'), recursive=True)):
    G8.update(pickle.load(open(f, 'rb')))
need = sorted(set(DEV[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in need if c not in G8]
assert not miss, f"{len(miss)} clips missing from G8a, e.g. {miss[:3]}"


def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


# per-clip frame features except the PCA part (18 dims), and the raw HSEmotion embeddings
FB, EMB = {}, {}
for c in need:
    fs = G8[c]['faces']
    if not fs:
        FB[c], EMB[c] = np.zeros((0, 18), np.float32), None
        continue
    dur = max(G8[c]['meta'].get('duration') or 0.0, 1e-3)
    fer = np.stack([d['fer'] for d in fs]).astype(np.float32)
    box = np.stack([d['box'] for d in fs])
    FB[c] = np.concatenate([
        softmax(fer[:, :8]), fer[:, 8:10], np.stack([d['pose'] for d in fs]) / 90.0,
        np.stack([(box[:, 0] + box[:, 2]) / 2, (box[:, 1] + box[:, 3]) / 2,
                  np.sqrt(np.clip((box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]), 0, None))], 1),
        np.nan_to_num(np.array([[d['mouth'] * 10] for d in fs], np.float32)),
        np.array([[min(d['t'] / dur, 1.0)] for d in fs], np.float32)], 1).astype(np.float32)
    EMB[c] = np.stack([d['fer_emb'] for d in fs]) if all('fer_emb' in d for d in fs) else None
HAS_EMB = all(EMB[c] is not None for c in need if len(G8[c]['faces']))
FDIM = (PCA_DIM if HAS_EMB else 0) + 18
print(f"HSEmotion embedding available: {HAS_EMB} | frame feature dim {FDIM}")


def pick_frames(idx, cap):
    return idx if len(idx) <= cap else [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]


def sync(c, face_ids):
    # correlation of mouth opening with the audio energy envelope, and mouth variability, for a set of faces
    A = G8[c]['audio']
    fs = [G8[c]['faces'][j] for j in face_ids]
    fs = [d for d in fs if np.isfinite(d['mouth'])]
    if A is None or len(fs) < 4:
        return 0.0, 0.0
    m = np.array([d['mouth'] for d in fs])
    env = A['env']
    e = np.array([env[min(int(d['t'] * 10), len(env) - 1)] for d in fs]) if len(env) else np.zeros(len(fs))
    r = float(np.corrcoef(m, e)[0, 1]) if m.std() > 1e-6 and e.std() > 1e-9 else 0.0
    return r, float(m.std() * 10)


def mean12(c_faces, n_sampled):
    if not c_faces:
        return np.zeros(12, np.float32)
    fer = np.stack([d['fer'] for d in c_faces]).astype(np.float32)
    v = np.concatenate([softmax(fer[:, :8]), fer[:, 8:10]], 1).mean(0)
    return np.concatenate([v, [1.0, len({d['frame'] for d in c_faces}) / max(n_sampled, 1)]]).astype(np.float32)


NVOICE = 3 * 2 + 3
SLOT, PSLOT = {}, {}                      # (n, role, clip) / (n, clip) -> (clip id, face indices)
FMASK = np.zeros((N, 3, 3, MAXF), bool); PMASK = np.zeros((N, 1, 3, MAXF_POOL), bool)
VOI = np.zeros((N, 3, NVOICE), np.float32)
LRF = np.zeros((N, 60), np.float32)       # G6b-style role means (late fusion and the joint-branch arm)
CENT_COS = np.full(N, np.nan, np.float32)
for n, row in enumerate(tqdm(DEV.itertuples(), total=N, desc='roles')):
    cl = [row.clip1, row.clip2, row.clip3]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8[c]['faces']))]
    lab = np.zeros(len(items), int)
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else None
    if len(items) > 1:
        lab = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                      distance_threshold=1 - SAME_PERSON_COS).fit_predict(E)
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    if A is not None and L is not None:
        ca, cb = E[lab == A].mean(0), E[lab == L].mean(0)
        CENT_COS[n] = float(ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb) + 1e-9))
    role = lambda p: 0 if p == A else (1 if p == L else 2)
    by = defaultdict(list)
    for (k, j), p in zip(items, lab):
        by[(role(p), k)].append(j)
        by[('pool', k)].append(j)
    for k, c in enumerate(cl):
        for r in range(3):
            idx = pick_frames(sorted(by[(r, k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF)
            SLOT[(n, r, k)] = (c, idx); FMASK[n, r, k, :len(idx)] = True
        idx = pick_frames(sorted(by[('pool', k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF_POOL)
        PSLOT[(n, k)] = (c, idx); PMASK[n, 0, k, :len(idx)] = True
        v = [x for r in range(3) for x in sync(c, by[(r, k)])]
        a3, ak = G8[cl[2]]['audio'], G8[c]['audio']
        ok = a3 is not None and ak is not None and a3.get('ecapa') is not None and ak.get('ecapa') is not None
        vcos = float(a3['ecapa'].astype(np.float32) @ ak['ecapa'].astype(np.float32)) if ok else 0.0
        VOI[n, k] = v + [vcos, float(ok), len(set(lab)) / 5.0]

    def faces_of(k, p):
        return [G8[cl[k]]['faces'][j] for (kk, j), q in zip(items, lab) if kk == k and q == p]

    def dominant(k):
        ns = G8[cl[k]]['meta']['n_sampled']
        cand = sorted({q for (kk, q) in frames if kk == k}, key=lambda q: -len(frames[(k, q)]))
        return cand[0] if cand and len(frames[(k, cand[0])]) / max(ns, 1) >= DOMINANT_MIN_FRAC else None

    n3 = G8[cl[2]]['meta']['n_sampled']
    blocks = [mean12(faces_of(2, A) if A is not None else [], n3), mean12(faces_of(2, L) if L is not None else [], n3),
              mean12((faces_of(0, L) + faces_of(1, L)) if L is not None else [], n3)]
    for k in (1, 0):
        d = dominant(k)
        blocks.append(mean12(faces_of(k, d) if d is not None else [], G8[cl[k]]['meta']['n_sampled']))
    LRF[n] = np.concatenate(blocks)

VIS = FMASK[:, 1, 2].any(-1)
print(f"A found in III {FMASK[:, 0, 2].any(-1).mean() * 100:.1f}% | listener visible in III {VIS.mean() * 100:.1f}% | "
      f"listener also in I/II {FMASK[:, 1, :2].any((-1, -2)).mean() * 100:.1f}%")


def build_face_tensors(fit_clips):
    # PCA of the HSEmotion embedding fitted on faces of the training clips of the current fold only
    pca, var = None, None
    if HAS_EMB:
        pool = np.concatenate([EMB[c] for c in fit_clips if EMB.get(c) is not None])
        pick = np.random.default_rng(0).choice(len(pool), min(60000, len(pool)), replace=False)
        pca = PCA(PCA_DIM, random_state=0).fit(pool[pick].astype(np.float32))
        var = float(pca.explained_variance_ratio_.sum())
    FV = {}
    for c in need:
        if len(FB[c]) == 0:
            FV[c] = np.zeros((0, FDIM), np.float32)
        else:
            FV[c] = np.concatenate([pca.transform(EMB[c].astype(np.float32)), FB[c]], 1) if HAS_EMB else FB[c]
    Fa = np.zeros((N, 3, 3, MAXF, FDIM), np.float16)
    Pa = np.zeros((N, 1, 3, MAXF_POOL, FDIM), np.float16)
    for (n, r, k), (c, idx) in SLOT.items():
        if idx:
            Fa[n, r, k, :len(idx)] = FV[c][idx]
    for (n, k), (c, idx) in PSLOT.items():
        if idx:
            Pa[n, 0, k, :len(idx)] = FV[c][idx]
    return torch.tensor(Fa, device=DEVICE), torch.tensor(Pa, device=DEVICE), var

## Models

In [ ]:
T = lambda a, dt=None: torch.tensor(a, device=DEVICE) if dt is None else torch.tensor(a, dtype=dt, device=DEVICE)
FMASK, PMASK, VOI = T(FMASK), T(PMASK), T(VOI)
FACE = POOL = LRFZ = None             # set per fold
CLIPIDX = T([[CIDX[c] for c in r] for r in DEV[['clip1', 'clip2', 'clip3']].values])
TXT, AUD, AFD = FEAT['text'][CLIPIDX], F.normalize(FEAT['audio'][CLIPIDX], dim=-1), FEAT['afound'][CLIPIDX].float()
fm = FEAT['fmask'][CLIPIDX].unsqueeze(-1).float()
SCN = torch.cat([FEAT['ori'][CLIPIDX].mean(2), (FEAT['face'][CLIPIDX] * fm).sum(2) / fm.sum(2).clamp(min=1)], -1)
ZREC = torch.zeros(N, 3, N_REC, device=DEVICE)
YB, YA = T(DEV.yB.values), T(DEV.yA.values)


class FramePool(nn.Module):
    def __init__(self, fin, d):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(fin), nn.Linear(fin, d), nn.GELU(), nn.Linear(d, d))
        self.score = nn.Linear(d, 1)

    def forward(self, x, m):                      # x [..., F, fin], m [..., F]
        h = self.proj(x.float())
        a = self.score(h).squeeze(-1).masked_fill(~m, -1e4)
        w = torch.softmax(a, -1) * m.float()
        return (w.unsqueeze(-1) * h).sum(-2), m.any(-1)


class RoleNet(nn.Module):
    def __init__(self, cfg, d=RN['D']):
        super().__init__()
        self.cfg, self.R = cfg, (3 if cfg['role'] else 1)
        if cfg['faces']:
            self.pool = FramePool(FDIM, d)
            self.absent = nn.Parameter(torch.randn(self.R, 3, d) * 0.02)
            self.face_role = nn.Parameter(torch.randn(self.R, d) * 0.02)
        if cfg['ctx']:
            self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
            self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
            self.voice = nn.Linear(NVOICE, d)
            self.scene = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, d))
            self.ctx_role = nn.Parameter(torch.randn(2, d) * 0.02)
        self.clip_emb = nn.Parameter(torch.randn(3, d) * 0.02)
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, RN['heads'], 4 * d, RN['dropout'], batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, RN['layers'], enable_nested_tensor=False)
        mk = lambda: nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 7))
        self.head = mk()
        self.head_face = mk() if cfg['aux'] and cfg['faces'] else None
        self.head_ctx = mk() if cfg['aux'] and cfg['ctx'] else None
        self.head_A = mk() if cfg['aux'] and cfg['faces'] and cfg['role'] else None

    def forward(self, ix, train=False):
        B, groups, aux = len(ix), [], {}
        if self.cfg['faces']:
            x, m = (FACE[ix], FMASK[ix]) if self.R == 3 else (POOL[ix], PMASK[ix])
            h, present = self.pool(x, m)                                      # [B, R, 3, d]
            h = torch.where(present.unsqueeze(-1), h, self.absent.unsqueeze(0).expand(B, -1, -1, -1))
            h = h + self.face_role[None, :, None] + self.clip_emb[None, None]
            ft = h.reshape(B, self.R * 3, -1)
            groups.append(ft)
            if self.head_face is not None:
                aux['face'] = (self.head_face(ft.mean(1)), YB[ix], RN['aux_w'])
            if self.head_A is not None:
                tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
                aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        if self.cfg['ctx']:
            spk_ = self.text(TXT[ix]) + self.audio(AUD[ix]) * AFD[ix].unsqueeze(-1) + self.voice(VOI[ix]) + self.ctx_role[0]
            scn = self.scene(SCN[ix]) + self.ctx_role[1]
            ct = torch.cat([spk_ + self.clip_emb, scn + self.clip_emb], 1)     # [B, 6, d]
            groups.append(ct)
            if self.head_ctx is not None:
                aux['ctx'] = (self.head_ctx(ct.mean(1)), YB[ix], RN['aux_w'])
        toks = torch.cat([self.query.expand(B, -1, -1)] + groups, 1)
        valid = torch.ones(toks.shape[:2], dtype=torch.bool, device=toks.device)
        if train and self.cfg['mdrop'] and len(groups) == 2:
            u = torch.rand(B, device=toks.device)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            nf = groups[0].shape[1]
            valid[:, 1:1 + nf] &= ~drop_face.unsqueeze(1)
            valid[:, 1 + nf:] &= ~drop_ctx.unsqueeze(1)
        out = self.enc(toks, src_key_padding_mask=~valid)
        return self.head(out[:, 0]), aux


class B1Wrap(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False)

    def forward(self, ix, train=False):
        return self.f(CLIPIDX[ix], ZREC[ix]), {}


class B1FaceWrap(nn.Module):
    # B1 + the 60-d G6b-style role-face vector through a zero-initialised branch, trained jointly (the G7b design)
    def __init__(self, n_face=60, d=512):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False, d=d)
        self.face = nn.Sequential(nn.Linear(n_face, d // 2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d // 2, d))
        nn.init.zeros_(self.face[-1].weight); nn.init.zeros_(self.face[-1].bias)

    def forward(self, ix, train=False):
        b, clip_idx = self.f, CLIPIDX[ix]
        B, n = clip_idx.shape
        feats = gather(clip_idx)
        tok = b.enc({k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}).reshape(B, n, -1)
        h = b.inter(tok + b.clip_pos[:, 3 - n:])
        return b.head(h.mean(1) + self.face(LRFZ[ix])), {}


HP = {'b1': dict(lr=LR, wd=WEIGHT_DECAY, epochs=FC_EPOCHS, patience=PATIENCE, batch=FC_BATCH),
      'role': dict(lr=RN['lr'], wd=RN['wd'], epochs=RN['epochs'], patience=RN['patience'], batch=RN['batch'])}
HP['b1face'] = HP['b1']
MAKE = {'b1': lambda cfg: B1Wrap(), 'b1face': lambda cfg: B1FaceWrap(), 'role': lambda cfg: RoleNet(cfg)}
print("parameters:", {n: f"{sum(p.numel() for p in MAKE[k](c).parameters()) / 1e6:.2f}M" for n, k, c in EXPERIMENTS})


def predict(model, ix, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ix), bs):
            out.append(F.softmax(model(ix[i:i + bs])[0], -1).cpu())
    return torch.cat(out).numpy()


def train_eval(kind, cfg, tr, dev, te, seed):
    seed_all(seed)
    hp = HP[kind]
    model = MAKE[kind](cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    y_dev = YB[dev].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(hp['epochs']):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), hp['batch']):
            j = perm[i:i + hp['batch']]
            logits, aux = model(j, train=True)
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        u = war_uar(predict(model, dev).argmax(1), y_dev, 7)[1]
        if u > best:
            best, bad = u, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= hp['patience']:
                break
    model.load_state_dict(best_state)
    return predict(model, te), best

## PaperBest: the Hi-EF paper's best configuration (verbatim from the replication notebook)

In [ ]:
class TemporalTransformer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_layers=2, dropout=0.1):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, 16, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
                                           dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)

    def forward(self, x):
        return self.transformer(x + self.pos_encoding[:, :x.size(1), :])


class CrossAttentionFusion(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_layers=1, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
                                     for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])

    def forward(self, query, key_values):
        x = query
        for attn, norm in zip(self.layers, self.norms):
            attended, _ = attn(x, key_values, key_values)
            x = norm(x + attended)
        return x


class IntraVideoFusion(nn.Module):
    def __init__(self, d_model=512, audio_dim=527):
        super().__init__()
        self.face_temporal = TemporalTransformer(d_model, n_heads=8, n_layers=2)
        self.ori_temporal = TemporalTransformer(d_model, n_heads=8, n_layers=2)
        self.type_fusion = CrossAttentionFusion(d_model, n_heads=8, n_layers=1)
        self.audio_proj = nn.Linear(audio_dim, d_model)
        self.modality_fusion = CrossAttentionFusion(d_model, n_heads=8, n_layers=1)

    def forward(self, face_features, ori_features, text_feature, audio_feature):
        face_out = self.face_temporal(face_features).mean(dim=1, keepdim=True)
        ori_out = self.ori_temporal(ori_features).mean(dim=1, keepdim=True)
        video_feat = self.type_fusion(face_out, torch.cat([face_out, ori_out], dim=1))
        audio_feat = self.audio_proj(F.normalize(audio_feature, dim=-1)).unsqueeze(1)
        text_feat = text_feature.unsqueeze(1)
        clip_feat = self.modality_fusion(video_feat, torch.cat([video_feat, text_feat, audio_feat], dim=1))
        return clip_feat.squeeze(1)


class InterVideoFusion(nn.Module):
    def __init__(self, d_model=512, lstm_layers=3, transformer_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=d_model, hidden_size=d_model, num_layers=lstm_layers, batch_first=False,
                            dropout=0.1)
        self.pos_encoding = nn.Parameter(torch.randn(1, 3, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=8, dim_feedforward=d_model * 4, dropout=0.1,
                                           batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=transformer_layers, enable_nested_tensor=False)

    def forward(self, c1, c2, c3):
        lstm_out, _ = self.lstm(torch.stack([c1, c2, c3], dim=0))
        return self.transformer(lstm_out.permute(1, 0, 2) + self.pos_encoding).mean(dim=1)


class PaperBest(nn.Module):
    def __init__(self, d_model=512, n_classes=7):
        super().__init__()
        self.intra_fusion = IntraVideoFusion(d_model)
        self.inter_fusion = InterVideoFusion(d_model)
        self.classifier = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(0.3), nn.Linear(d_model, d_model // 2),
                                        nn.GELU(), nn.Dropout(0.2), nn.Linear(d_model // 2, n_classes))

    def forward(self, ix, train=False):
        feats = gather(CLIPIDX[ix])
        clips = [self.intra_fusion(feats['face'][:, k], feats['ori'][:, k], feats['text'][:, k], feats['audio'][:, k])
                 for k in range(3)]
        return self.classifier(self.inter_fusion(*clips)), {}


print(f"PaperBest parameters: {sum(p.numel() for p in PaperBest().parameters()):,}")


def train_paper(tr, dev, te, seed):
    seed_all(seed)
    model = PaperBest().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=5, factor=0.5)
    y_dev = YB[dev]
    best, best_state = -1, None
    for ep in range(PAPER_EPOCHS):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), PAPER_BATCH):
            j = perm[i:i + PAPER_BATCH]
            loss = F.cross_entropy(model(j)[0], YB[j])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        p = predict(model, dev)
        sched.step(F.nll_loss(torch.log(torch.tensor(p) + 1e-9), y_dev.cpu()).item())
        u = war_uar(p.argmax(1), y_dev.cpu().numpy(), 7)[1]
        if u > best:
            best = u
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return predict(model, te), best

## Train on the 45 train+val episodes, evaluate once on test

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

sel_eps = sorted(random.Random(SELECT_SEED).sample(list(EPS), N_INNER_DEV))
trr = np.where(~IS_TEST)[0]
fit_rows = np.where(~IS_TEST & ~np.isin(src, sel_eps))[0]
dev_rows = np.where(np.isin(src, sel_eps))[0]
te_rows = np.where(IS_TEST)[0]
assert not set(src[te_rows]) & set(src[trr])
print(f"fit {len(fit_rows)} | selection {len(dev_rows)} (episodes {sel_eps}) | test {len(te_rows)}")
y_all = DEV.yB.values
FACE, POOL, var = build_face_tensors(sorted(set(DEV.iloc[trr][['clip1', 'clip2', 'clip3']].values.ravel())))
mu, sd = LRF[trr].mean(0), LRF[trr].std(0) + 1e-6
LRFZ = T(((LRF - mu) / sd).astype(np.float32))
LOGPI_TR = np.log((np.bincount(y_all[trr], minlength=7) + 1) / (len(trr) + 7))
tr, dev, te = T(fit_rows), T(dev_rows), T(te_rows)

PROBS, log = {}, []
t0 = time.time()
for name, kind, cfg in EXPERIMENTS + [("PaperBest", 'paper', None)]:
    PROBS[name] = []
    for seed in SEEDS:
        p, sel = train_paper(tr, dev, te, seed) if kind == 'paper' else train_eval(kind, cfg, tr, dev, te, seed)
        PROBS[name].append(p)
        w, u = war_uar(p.argmax(1), y_all[te_rows], 7)
        log.append({'exp': name, 'seed': seed, 'sel_UAR': sel, 'test_UAR': u, 'test_WAR': w})
        print(f"{name:<15} seed {seed}: sel {sel:5.2f} | test UAR {u:5.2f} WAR {w:5.2f} | {(time.time() - t0) / 60:.1f} min",
              flush=True)
        torch.cuda.empty_cache()
    PROBS[name] = np.stack(PROBS[name])

sc = StandardScaler().fit(LRF[trr])
Xtr, Xte = sc.transform(LRF[trr]), sc.transform(LRF[te_rows])
best = None
for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1]:
    s = [war_uar(LogisticRegression(max_iter=3000, C=C).fit(Xtr[a], y_all[trr][a]).predict(Xtr[b]), y_all[trr][b], 7)[1]
         for a, b in GroupKFold(5).split(Xtr, y_all[trr], src[trr])]
    if best is None or np.mean(s) > best[0]:
        best = (np.mean(s), C)
clf = LogisticRegression(max_iter=3000, C=best[1]).fit(Xtr, y_all[trr])
pf = np.full((len(te_rows), 7), 1e-6, np.float32); pf[:, clf.classes_] = clf.predict_proba(Xte); pf /= pf.sum(1, keepdims=True)
PROBS['LateFusion'] = np.stack([np.exp((1 - LATE_W) * np.log(PROBS['B1'][s] + 1e-9) + LATE_W * np.log(pf + 1e-9))
                                for s in range(len(SEEDS))])
PROBS['FaceLR'] = pf[None]
pd.DataFrame(log).to_csv(f"{OUT_DIR}/g10_test_per_seed.csv", index=False)
np.savez(f"{OUT_DIR}/g10_test_probs.npz", sample_id=DEV.sample_id.values[te_rows], logpi=LOGPI_TR,
         **{k.replace('-', '_'): v for k, v in PROBS.items()})

## Results: preregistered fixed-sequence contrasts, then descriptive analyses

In [ ]:
yt, st = y_all[te_rows], src[te_rows]
vis_t = VIS[te_rows]
FEAR = E2I['fear']


def recalls(p, y):
    return np.array([(p[y == c] == c).mean() * 100 if (y == c).any() else np.nan for c in range(7)])


def uar7(p, y):
    return np.nanmean(recalls(p, y))


def uar6(p, y):
    return np.nanmean(np.delete(recalls(p, y), FEAR))


LOGP = {k: np.log(v.mean(0) + 1e-9) for k, v in PROBS.items()}
PRED = {'LA': {k: (v - LA_TAU * LOGPI_TR).argmax(1) for k, v in LOGP.items()}, 'plain': {k: v.argmax(1) for k, v in LOGP.items()}}
rng = np.random.default_rng(0)
G = [np.where(st == e)[0] for e in np.unique(st)]
BOOT = [np.concatenate([G[j] for j in rng.integers(0, len(G), len(G))]) for _ in range(2000)]


def contrast(a, b, mode='LA', metric=uar7, m=None):
    m = np.ones(len(yt), bool) if m is None else m
    pa, pb = PRED[mode][a], PRED[mode][b]
    d0 = metric(pa[m], yt[m]) - metric(pb[m], yt[m])
    ds = [metric(pa[i[m[i]]], yt[i[m[i]]]) - metric(pb[i[m[i]]], yt[i[m[i]]]) for i in BOOT]
    lo, hi = np.nanpercentile(ds, [2.5, 97.5])
    return d0, lo, hi


print(f"test MCIS {len(yt)} | episodes {len(G)} | class counts {dict(zip(EMO, np.bincount(yt, minlength=7)))}")
print("\n== per-seed test UAR (plain) ==")
print(pd.DataFrame(log).pivot(index='seed', columns='exp', values='test_UAR').round(2).to_string())
print("\n== seed ensemble (LA | plain), with 95% bootstrap over test episodes for LA UAR ==")
for k in PRED['LA']:
    ds = [uar7(PRED['LA'][k][i], yt[i]) for i in BOOT]
    lo, hi = np.nanpercentile(ds, [2.5, 97.5])
    print(f"  {k:<15} UAR LA {uar7(PRED['LA'][k], yt):5.2f} [{lo:5.1f},{hi:5.1f}]  WAR LA {(PRED['LA'][k] == yt).mean() * 100:5.2f} | "
          f"UAR plain {uar7(PRED['plain'][k], yt):5.2f}  WAR plain {(PRED['plain'][k] == yt).mean() * 100:5.2f} | "
          f"6-class LA {uar6(PRED['LA'][k], yt):5.2f}")
print("  (paper, different split, not comparable: UAR 23.72, WAR 35.19)")

print("\n== PREREGISTERED fixed-sequence contrasts (ΔUAR, LA, 7-class, seed ensemble) ==")
alive = True
for i, (a, b) in enumerate([("RoleNet", "PaperBest"), ("RoleNet", "B1"), ("RoleNet", "LateFusion"),
                            ("RoleNet", "RoleNet-noRole")], 1):
    d, lo, hi = contrast(a, b)
    ok = lo > 0
    status = ('CONFIRMED' if ok else 'NOT CONFIRMED') if alive else 'descriptive only (sequence stopped)'
    print(f"  {i}. {a} − {b}: ΔUAR {d:+.2f} [{lo:+.2f},{hi:+.2f}] -> {status}")
    alive = alive and ok

print("\n== descriptive ==")
for a, b in [("RoleNet", "PaperBest"), ("RoleNet", "B1"), ("RoleNet", "LateFusion"), ("RoleNet", "RoleNet-noRole"),
             ("PaperBest", "B1")]:
    r = [contrast(a, b, 'LA', uar6), contrast(a, b, 'plain', uar7),
         contrast(a, b, 'LA', uar7, vis_t), contrast(a, b, 'LA', uar7, ~vis_t)]
    print(f"  {a:<9}− {b:<15} 6-class LA {r[0][0]:+5.2f} [{r[0][1]:+.1f},{r[0][2]:+.1f}] | plain {r[1][0]:+5.2f} "
          f"[{r[1][1]:+.1f},{r[1][2]:+.1f}] | listener visible {r[2][0]:+5.2f} (n={vis_t.sum()}) | not visible {r[3][0]:+5.2f} (n={(~vis_t).sum()})")
print("\n== per-class recall (LA) ==")
print(f"  {'':<15}" + "".join(f"{e[:7]:>8}" for e in EMO))
for k, p in PRED['LA'].items():
    print(f"  {k:<15}" + "".join(f"{x:8.1f}" for x in recalls(p, yt)))
print("\n== per-episode UAR (LA): RoleNet vs PaperBest vs B1 ==")
wins = 0
for e in np.unique(st):
    m = st == e
    r, pb_, b1 = uar7(PRED['LA']['RoleNet'][m], yt[m]), uar7(PRED['LA']['PaperBest'][m], yt[m]), uar7(PRED['LA']['B1'][m], yt[m])
    wins += r > pb_
    print(f"  episode {e} (n={m.sum()}): RoleNet {r:5.2f} | PaperBest {pb_:5.2f} | B1 {b1:5.2f}")
print(f"  RoleNet beats PaperBest in {wins}/{len(np.unique(st))} episodes")